In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from injest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag("How do I run Olama locally?")
print (answer)

I don’t see a FAQ entry for **Olama/Ollama** specifically.

If you mean running the course **locally**, the FAQ says you can do that instead of Codespaces if you’re comfortable setting up:

- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

If you run locally, make sure to **document your setup** and keep your environment **reproducible**.

If you meant **Ollama** specifically, please share the relevant module or FAQ entry and I can check whether it’s covered.


In [5]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—usually you can, but it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you check:\n- whether the course is still accepting students\n- the enrollment deadline\n- any prerequisites or restrictions\n- how to register\n\nIf you share the course name or a link, I can help you figure it out.'

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search("How do I run Ollama?")

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join after the course has started? discovered the course join late enrollment"}', call_id='call_iHUbdBtDc4rZmASnKQJGRawq', name='search', type='function_call', id='fc_0e379cdab37d1723006a406e3db88c819d8aea87d929fbbf1e', namespace=None, status='completed')]

In [10]:

response.output[0]

ResponseFunctionToolCall(arguments='{"query":"Can I join after the course has started? discovered the course join late enrollment"}', call_id='call_iHUbdBtDc4rZmASnKQJGRawq', name='search', type='function_call', id='fc_0e379cdab37d1723006a406e3db88c819d8aea87d929fbbf1e', namespace=None, status='completed')

In [11]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [12]:
messages.extend(response.output)
print(messages)
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})
print(messages)

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}, ResponseFunctionToolCall(arguments='{"query":"Can I join after the course has started? discovered the course join late enrollment"}', call_id='call_iHUbdBtDc4rZmASnKQJGRawq', name='search', type='function_call', id='fc_0e379cdab37d1723006a406e3db88c819d8aea87d929fbbf1e', namespace=None, status='completed')]
[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}, ResponseFunctionToolCall(arguments='{"query":"Can I join after the course has started? discovered the course join late enrollment"}', call_id='call_iHUbdBtDc4rZmASnKQJGRawq', name='search', type='function_call', id='fc_0e379cdab37d1723006a406e3db88c819d8aea87d929fbbf1e', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_iHUbdBtDc4rZmASnKQJGRawq', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I jus

In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join and start learning anytime.\n\nIf you want a certificate, though, you need to submit your project while the course is still accepting submissions.'

In [14]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(776, 37)

In [15]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

In [16]:
result = calculate_gpt54mini_price(772, 32)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.000135


In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [33]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [34]:
response.output

[ResponseOutputMessage(id='msg_0be66ad43df42315006a407364db40819fb302da6aa7ddf436', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open. You can also start learning and working through the materials even if you discovered it late.\n\nIf you’d like, I can also help with:\n- how to start the course\n- whether you can still get a certificate\n- the weekly workflow / deadlines\n\nIs there anything else you want to explore?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [30]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [35]:
messages.extend(response.output)

for item in response.output:
    if item.type == "function_call":
        print("function call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)


ASSISTANT:
Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still open. You can also start learning and working through the materials even if you discovered it late.

If you’d like, I can also help with:
- how to start the course
- whether you can still get a certificate
- the weekly workflow / deadlines

Is there anything else you want to explore?


In [41]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course enrollment discovered the course can I join"}', call_id='call_IMZZMDOsluoku211QUqIRJWf', name='search', type='function_call', id='fc_053586595dff798b006a4077d2f5d4819da628df00a7774548', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_IMZZMDOsluoku211QUqIRJWf',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "cour

In [50]:
def agent_loop(instructions, questions, model='gpt-5.4-mini') -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": questions},
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool],
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [46]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [51]:
agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course discovered can I join course enrollment late registration"}
iteration #2...
ASSISTANT:


'Yes — you can still join the course. If you want to receive a certificate, you need to submit your project while the course is still accepting submissions.\n\nWould you like help with anything else about the course?'

In [52]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [54]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [55]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [56]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [59]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [60]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [64]:
result = runner.loop(
    prompt='How do I run Olma locally?',
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [65]:
result.cost

CostInfo(input_cost=Decimal('0.00275925'), output_cost=Decimal('0.001143'), total_cost=Decimal('0.00390225'))

In [66]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olma locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olma run locally local setup local development how to run Olma 

In [67]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [68]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='how do i run docker in docker?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"docker in docker run docker in docker